# Notebook 04 — Richer Noise Models & the Detection Challenge

In Notebooks 01–03 we worked with **depolarizing noise**: each timestep's error
rate was independent of every other.  Real quantum hardware is messier.

This notebook introduces two richer noise models:

| Model | Physical source | Key feature |
|---|---|---|
| **Dephasing** | T2 decoherence, phase scrambling | Bit-flip rate bounded in [0, 0.5] |
| **Correlated (AR(1))** | Slow environmental drift, non-Markovian memory | Current noise depends on past noise |

The central finding: **correlated noise is harder to detect** with standard
methods, and **autocorrelation-based features help close the gap**.

---

**PhD connection — Giarmatzi / Tonekaboni group:**
Their research characterises *how* noise is correlated over time
(non-Markovian noise).  But before you can characterise the correlation
structure, you first need to know *when* the correlation regime changed.  
This notebook builds the detection tool that answers that prior question.

**PhD connection — Sanders / Usman (CSIRO) group:**
If the device drifts *gradually* (correlated drift rather than a sharp jump),
standard detectors may miss it entirely — meaning your verification experiment
collects data from two different noise regimes without knowing it.  The rolling
autocorrelation feature detects this kind of subtle, slow drift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from quantum_drift_detector.simulators import (
    DepolarizingSimulator,
    DephasingSimulator,
    CorrelatedNoiseSimulator,
)
from quantum_drift_detector.features import (
    extract_bit_flip_rate,
    extract_entropy,
    extract_rolling_mean,
    extract_rolling_variance,
    extract_rolling_autocorr,
)
from quantum_drift_detector.detectors import run_cusum, run_window_detector

np.random.seed(42)

N_TIMESTEPS = 150
N_SHOTS     = 1_000
CHANGEPOINT = 75

## Part 1 — Dephasing Noise

### The Ramsey experiment (plain English)

Imagine a coin that spins perfectly upright (state = 0).  Now tip it slightly
onto its edge (quantum superposition = |+⟩).  In this position:
- **Bit-flip noise** (depolarizing) would slap it over to tails.
- **Dephasing noise** scrambles the *direction* of the tipping without
  flipping the coin.  If you then gently tip it back to upright and look,
  the scrambling shows up as "tails" outcomes.

This is the **Ramsey experiment**: |0⟩ → H → [noise] → H → measure.

**Key physical fact:**
$$P(\text{measure } 1 \mid \text{dephasing rate } \gamma) = \frac{1 - e^{-\gamma}}{2}$$

The bit-flip rate is **bounded in [0, 0.5]** — dephasing can make the qubit
at most 50% random (fully incoherent), it cannot flip it past that.

In [ ]:
# Dephasing rate doubles at the changepoint: γ: 0.2 → 1.5
# p_pre  = (1 - exp(-0.2)) / 2 ≈ 0.09
# p_post = (1 - exp(-1.5)) / 2 ≈ 0.39

deph_sim = DephasingSimulator(gamma_pre=0.2, gamma_post=1.5, changepoint=CHANGEPOINT)
deph_data = deph_sim.generate_data(N_TIMESTEPS, N_SHOTS)
deph_rates = extract_bit_flip_rate(deph_data)

print(f"Dephasing simulator")
print(f"  gamma_pre  = {deph_sim.gamma_pre}  →  P(1) = {deph_sim.p_pre:.4f}")
print(f"  gamma_post = {deph_sim.gamma_post}  →  P(1) = {deph_sim.p_post:.4f}")
print(f"  Observed mean rate pre : {deph_rates[:CHANGEPOINT].mean():.4f}")
print(f"  Observed mean rate post: {deph_rates[CHANGEPOINT:].mean():.4f}")
print(f"  Note: both values are ≤ 0.5 (bounded by the Ramsey physics)")

In [ ]:
# Compare depolarizing vs. dephasing side by side.
depol_sim = DepolarizingSimulator(error_rate_pre=0.09, error_rate_post=0.39,
                                  changepoint=CHANGEPOINT)
depol_data = depol_sim.generate_data(N_TIMESTEPS, N_SHOTS)
depol_rates = extract_bit_flip_rate(depol_data)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
t = np.arange(N_TIMESTEPS)

for ax, rates, title, color in [
    (axes[0], depol_rates, 'Depolarizing noise (P(1) up to 1.0)', 'steelblue'),
    (axes[1], deph_rates,  'Dephasing noise (P(1) bounded ≤ 0.5)', 'darkorange'),
]:
    ax.plot(t, rates, color=color, alpha=0.8)
    ax.axvline(CHANGEPOINT, color='red', linestyle='--', label='True changepoint')
    ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='P(1) = 0.5 ceiling')
    ax.set(xlabel='Timestep', ylabel='Bit-flip rate', title=title, ylim=(-0.02, 1.02))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Depolarizing vs. Dephasing: different physical ceilings', fontsize=12)
plt.tight_layout()
plt.show()

print("Key observation: dephasing rates never exceed 0.5.")
print("This physical ceiling is a fingerprint of dephasing vs. depolarizing.")

## Part 2 — Correlated (non-Markovian) Noise

### What is temporal correlation in noise?

Think of the device temperature.  It doesn't jump randomly between −20 °C and
50 °C every second — it drifts slowly.  If it's warm now, it'll probably still
be warm in a few seconds.  This is **temporal correlation** (or "memory").

The same happens with quantum noise sources:
- Charge fluctuators near a superconducting qubit drift slowly.
- Laser intensity for a neutral-atom trap varies on a timescale of seconds.
- Magnetic field noise has a $1/f$ power spectrum (more energy at low frequencies).

We model this with an **AR(1) autoregressive process**:

$$\varepsilon_t = \mu_t + \varphi\,(\varepsilon_{t-1} - \mu_t) + \sigma\,\eta_t$$

where $\varphi \in [0, 1)$ is the **autocorrelation coefficient**:
- $\varphi = 0$: independent noise (Markovian — same as Phase 1).
- $\varphi = 0.9$: each timestep's error rate is 90% determined by the previous one.

At the changepoint, $\mu_t$ shifts from $\mu_\text{pre}$ to $\mu_\text{post}$.  
The AR(1) smoothing means the actual error rate **drifts gradually** toward the
new mean — there is no sharp jump for detectors to catch.

In [ ]:
# Generate correlated noise at three levels of autocorrelation.
params = [
    (0.0, 'steelblue',   'φ = 0.0  (Markovian)'),
    (0.6, 'darkorange',  'φ = 0.6  (moderate correlation)'),
    (0.9, 'crimson',     'φ = 0.9  (strong correlation)'),
]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, (phi, color, label) in zip(axes, params):
    np.random.seed(42)
    sim = CorrelatedNoiseSimulator(
        mu_pre=0.02, mu_post=0.08, phi=phi, sigma=0.008, changepoint=CHANGEPOINT
    )
    true_rates = sim.generate_error_rates(N_TIMESTEPS)  # latent process
    data = sim.generate_data(N_TIMESTEPS, N_SHOTS)
    observed_rates = extract_bit_flip_rate(data)

    ax.plot(observed_rates, color=color, alpha=0.5, label='Observed (shot noise)')
    ax.plot(true_rates, color=color, lw=2, label='True ε_t (latent)')
    ax.axvline(CHANGEPOINT, color='red', linestyle='--', lw=1.5, label='True changepoint')
    ax.axhline(0.02, color='gray', linestyle=':', lw=1)
    ax.axhline(0.08, color='gray', linestyle=':', lw=1)
    ax.set(ylabel='Error rate', title=label)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Timestep')
plt.suptitle('Effect of autocorrelation on noise appearance', fontsize=12)
plt.tight_layout()
plt.show()

print("Key observation:")
print("  φ = 0.0: sharp jump at t=75 — easy to detect.")
print("  φ = 0.9: smooth drift over many steps — hard to detect.")

## Part 3 — Why Standard Detectors Struggle with Correlated Noise

CUSUM was designed for **independent** observations.  When noise is correlated,
it accumulates false evidence: a run of slightly-high values (which are
naturally produced by AR(1) noise even *before* the changepoint) can
trigger a false alarm.  Conversely, the gradual transition after the changepoint
may not accumulate score fast enough to trigger a real alarm.

In [ ]:
N_TRIALS = 200
results = {}

for phi in [0.0, 0.6, 0.9]:
    delays = []
    false_alarms = 0

    for _ in range(N_TRIALS):
        # Real changepoint scenario
        np.random.seed(None)
        sim_cp = CorrelatedNoiseSimulator(
            mu_pre=0.02, mu_post=0.08, phi=phi, sigma=0.008, changepoint=CHANGEPOINT
        )
        rates_cp = extract_bit_flip_rate(sim_cp.generate_data(N_TIMESTEPS, N_SHOTS))
        res = run_cusum(rates_cp, target_mean=0.02, allowance=0.02, threshold=0.06)
        if res['detected_at'] is not None and res['detected_at'] >= CHANGEPOINT:
            delays.append(res['detected_at'] - CHANGEPOINT)

        # No changepoint (flat) scenario
        sim_flat = CorrelatedNoiseSimulator(
            mu_pre=0.02, mu_post=0.02, phi=phi, sigma=0.008, changepoint=CHANGEPOINT
        )
        rates_flat = extract_bit_flip_rate(sim_flat.generate_data(N_TIMESTEPS, N_SHOTS))
        res_flat = run_cusum(rates_flat, target_mean=0.02, allowance=0.02, threshold=0.06)
        if res_flat['detected_at'] is not None:
            false_alarms += 1

    results[phi] = {
        'avg_delay': np.mean(delays) if delays else float('nan'),
        'detection_rate': len(delays) / N_TRIALS,
        'false_alarm_rate': false_alarms / N_TRIALS,
    }

print(f"{'φ':>6}  {'Avg delay':>12}  {'Detection rate':>15}  {'False alarm rate':>18}")
print("-" * 60)
for phi, r in results.items():
    print(f"{phi:>6.1f}  {r['avg_delay']:>12.1f}  {r['detection_rate']:>15.2%}  {r['false_alarm_rate']:>18.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
phi_vals = list(results.keys())
colors = ['steelblue', 'darkorange', 'crimson']

axes[0].bar([str(p) for p in phi_vals], [results[p]['avg_delay'] for p in phi_vals],
            color=colors)
axes[0].set(xlabel='φ (autocorrelation)', ylabel='Avg detection delay (steps)',
            title='Detection delay vs. φ')

axes[1].bar([str(p) for p in phi_vals], [results[p]['detection_rate'] for p in phi_vals],
            color=colors)
axes[1].set(xlabel='φ', ylabel='Detection rate', title='Detection rate vs. φ', ylim=(0, 1))

axes[2].bar([str(p) for p in phi_vals], [results[p]['false_alarm_rate'] for p in phi_vals],
            color=colors)
axes[2].set(xlabel='φ', ylabel='False-alarm rate', title='False-alarm rate vs. φ', ylim=(0, 1))

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('CUSUM performance degrades as noise correlation increases', fontsize=11)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Higher φ → slower detection and more false alarms.")
print("  This is the core challenge of non-Markovian noise detection.")

## Part 4 — Autocorrelation as a Feature

The problem with the bit-flip rate alone: it measures the *mean level* of
noise, which drifts slowly after a changepoint in correlated noise.

A better signal: **the autocorrelation of the bit-flip rate**.  
After a changepoint, the noise transitions from one AR(1) process to another.
Even if the means are close, the *dynamics* (how quickly the process
reverts toward the mean) may change.  Rolling ACF(1) captures this.

More importantly: we can apply detectors to the *autocorrelation time series*
rather than the raw bit-flip rate — giving us sensitivity to structure changes
rather than just level changes.

In [ ]:
# Generate a scenario where phi changes at the changepoint (mean stays the same).
# This is a PURE correlation change — level-based detectors will completely miss it.
np.random.seed(42)

class MixedCorrelationSimulator:
    """Combines two AR(1) segments with different phi values."""
    def __init__(self, mu, phi_pre, phi_post, sigma, changepoint):
        self.sim_pre  = CorrelatedNoiseSimulator(mu_pre=mu, mu_post=mu, phi=phi_pre,
                                                  sigma=sigma, changepoint=9999)
        self.sim_post = CorrelatedNoiseSimulator(mu_pre=mu, mu_post=mu, phi=phi_post,
                                                  sigma=sigma, changepoint=9999)
        self.cp = changepoint
        self.mu = mu

    def generate_error_rates(self, n):
        r_pre  = self.sim_pre.generate_error_rates(self.cp)
        r_post = self.sim_post.generate_error_rates(n - self.cp)
        return np.concatenate([r_pre, r_post])

# phi: 0.1 → 0.9 at t=75 (same mean = 0.04)
mixed = MixedCorrelationSimulator(mu=0.04, phi_pre=0.1, phi_post=0.9,
                                   sigma=0.008, changepoint=CHANGEPOINT)
true_rates_mixed = mixed.generate_error_rates(N_TIMESTEPS)

# Features
rolling_mean  = extract_rolling_mean(true_rates_mixed, window=10)
rolling_acf   = extract_rolling_autocorr(true_rates_mixed, window=20, lag=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(true_rates_mixed, color='steelblue', lw=1.5)
axes[0].axvline(CHANGEPOINT, color='red', linestyle='--', label='True changepoint')
axes[0].set(ylabel='Error rate', title='Pure correlation change: φ = 0.1 → 0.9  (mean unchanged)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(rolling_mean, color='darkorange', lw=1.5)
axes[1].axvline(CHANGEPOINT, color='red', linestyle='--')
axes[1].set(ylabel='Rolling mean', title='Rolling mean — cannot see the changepoint')
axes[1].grid(True, alpha=0.3)

axes[2].plot(rolling_acf, color='crimson', lw=1.5)
axes[2].axvline(CHANGEPOINT, color='red', linestyle='--')
axes[2].axhline(0.0, color='gray', linestyle=':', lw=1)
axes[2].set(xlabel='Timestep', ylabel='Rolling ACF(1)',
            title='Rolling ACF(1) — reveals the correlation changepoint')
axes[2].grid(True, alpha=0.3)

plt.suptitle('ACF reveals what the mean cannot', fontsize=12)
plt.tight_layout()
plt.show()

## Part 5 — Detecting Correlated-Noise Changepoints Using ACF

Now apply the sliding-window KL detector to the rolling ACF time series
instead of the raw bit-flip rate.  This transforms the *correlation detection*
problem back into a *level detection* problem that our existing detectors handle well.

In [ ]:
# Full pipeline: correlated noise with both mean AND phi shifting.
np.random.seed(7)
sim_hard = CorrelatedNoiseSimulator(
    mu_pre=0.02, mu_post=0.05, phi=0.85, sigma=0.006, changepoint=CHANGEPOINT
)
data_hard = sim_hard.generate_data(N_TIMESTEPS, N_SHOTS)
rates_hard = extract_bit_flip_rate(data_hard)

# Feature 1: raw bit-flip rate
cusum_raw = run_cusum(rates_hard, target_mean=0.02, allowance=0.015, threshold=0.08)

# Feature 2: rolling ACF applied to bit-flip rate
acf_feature = extract_rolling_autocorr(rates_hard, window=20, lag=1)
# ACF is centred near 0.85 in both regimes but with different dynamics.
# Use the window detector on the ACF to find the structural break.
window_acf = run_window_detector(acf_feature, window_size=15, threshold=0.03)

print(f"True changepoint                      : t = {CHANGEPOINT}")
print(f"CUSUM on raw bit-flip rate            : t = {cusum_raw['detected_at']}")
print(f"Window detector on rolling ACF        : t = {window_acf['detected_at']}")

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
t = np.arange(N_TIMESTEPS)

axes[0].plot(t, rates_hard, color='steelblue', alpha=0.7, label='Bit-flip rate')
axes[0].axvline(CHANGEPOINT, color='red', linestyle='--', lw=1.5, label='True CP')
if cusum_raw['detected_at']:
    axes[0].axvline(cusum_raw['detected_at'], color='green', linestyle=':', lw=2,
                    label=f"CUSUM alarm t={cusum_raw['detected_at']}")
axes[0].set(ylabel='Bit-flip rate', title='Correlated noise (φ=0.85): raw signal')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, cusum_raw['scores'], color='darkorange', label='CUSUM score')
axes[1].axhline(cusum_raw['threshold'], color='purple', linestyle='--', lw=1)
axes[1].axvline(CHANGEPOINT, color='red', linestyle='--', lw=1.5)
axes[1].set(ylabel='Score', title='CUSUM score on raw bit-flip rate')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, acf_feature, color='crimson', alpha=0.8, label='Rolling ACF(1)')
axes[2].plot(t, window_acf['scores'] * 5, color='teal', lw=1.5,
             label='Window KL score on ACF (×5 for visibility)')
axes[2].axhline(window_acf['threshold'] * 5, color='purple', linestyle='--', lw=1)
axes[2].axvline(CHANGEPOINT, color='red', linestyle='--', lw=1.5)
if window_acf['detected_at']:
    axes[2].axvline(window_acf['detected_at'], color='teal', linestyle=':', lw=2,
                    label=f"ACF-window alarm t={window_acf['detected_at']}")
axes[2].set(xlabel='Timestep', ylabel='ACF / KL score',
            title='Rolling ACF(1) + window detector')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle('ACF-based detection on correlated noise', fontsize=12)
plt.tight_layout()
plt.show()

## Part 6 — Shannon Entropy as a Supporting Feature

Shannon entropy captures something neither the mean nor the ACF does well:
**whether the device is approaching the maximally-mixed state** (50/50 outcomes).

For depolarizing noise, entropy rises monotonically with the error rate.
For dephasing noise, entropy saturates at 1 bit (p = 0.5) — combining
entropy with the raw rate distinguishes the two noise types.

In [ ]:
np.random.seed(42)
entropy_depol = extract_entropy(depol_data)  # from Part 1
entropy_deph  = extract_entropy(deph_data)   # from Part 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, rates, entropy, label, color in [
    (axes[0], depol_rates, entropy_depol, 'Depolarizing', 'steelblue'),
    (axes[1], deph_rates,  entropy_deph,  'Dephasing',    'darkorange'),
]:
    ax.plot(rates,   color=color, alpha=0.6, label='Bit-flip rate')
    ax.plot(entropy, color='black', lw=2, linestyle='--', label='Entropy (bits)')
    ax.axvline(CHANGEPOINT, color='red', linestyle='--', lw=1.5, label='Changepoint')
    ax.set(xlabel='Timestep', ylabel='Value', title=f'{label}: rate vs entropy')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Entropy as a complementary feature', fontsize=12)
plt.tight_layout()
plt.show()

print("Key observation:")
print("  Depolarizing: entropy and bit-flip rate move together (both go to 1.0).")
print("  Dephasing: entropy saturates at 1 bit even as rate only reaches 0.5.")
print("  Using BOTH features together would help distinguish noise types.")

## Summary and PhD connections

| Finding | Implication |
|---|---|
| Dephasing produces bit-flip rates bounded at 0.5 | Noise *type* can be inferred from the signal ceiling |
| AR(1) noise makes changepoints look gradual | Standard CUSUM misses or delays detection |
| Rolling ACF detects correlation-structure changes | Even when the mean is unchanged, drift is visible |
| Entropy complements the raw rate | Helps distinguish depolarizing from dephasing |

**Giarmatzi / Tonekaboni:**
You now have a tool that detects *when* the non-Markovian regime changes.
Before characterising *how* noise is correlated, you can first segment the
time series at changepoints — giving clean data windows for each regime.

**Sanders / Usman (CSIRO):**
Correlated noise means drift can happen silently, without a sharp jump.
The rolling ACF flag provides an early warning even when the mean bit-flip
rate looks steady — ensuring you don't draw verification conclusions from
data that spans two different noise regimes.

---

**Next: Phase 4** — Ensemble detector combining CUSUM + BOCD + Window, and a
comprehensive evaluation across all noise types.